<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Parallel%20Cellular%20Algorithm%20(PCA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# 1. DEFINE PROBLEM & PARAMETERS
ROWS, COLS = 8, 8          # 8x8 rooftop grid
MAX_ITER = 30              # Number of iterations
SHADOW_PENALTY = 1.5       # Energy lost per adjacent panel causing shade
MAX_PANELS = 25            # Available installation space / budget limit

# Simulated sunlight intensity (kWh/day) across the rooftop (e.g., south-facing side is sunnier)
np.random.seed(42)
sunlight_map = np.random.uniform(4.0, 9.0, size=(ROWS, COLS))
sunlight_map[0:2, :] *= 0.5  # Simulate permanent chimney/tree shadow on top two rows

# 2. INITIALIZE POPULATION (0 = Empty roof spot, 1 = Solar Panel placed)
grid = np.zeros((ROWS, COLS), dtype=int)
initial_indices = np.random.choice(ROWS * COLS, MAX_PANELS, replace=False)
grid.flat[initial_indices] = 1

def get_neighbors(r, c):
    """Returns coordinates of Moore (8-neighbor) adjacent cells."""
    neighbors = []
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = r + dr, c + dc
            if 0 <= nr < ROWS and 0 <= nc < COLS:
                neighbors.append((nr, nc))
    return neighbors

def evaluate_cell(pos):
    """Calculates net energy score of a single rooftop cell."""
    r, c = pos
    neighbors = get_neighbors(r, c)
    # Count how many adjacent cells have panels (causes shading)
    adjacent_panels = sum(grid[nr, nc] for nr, nc in neighbors)

    # Potential energy if a panel sits at (r, c)
    net_energy = sunlight_map[r, c] - (SHADOW_PENALTY * adjacent_panels)
    return (r, c, net_energy)

def calculate_total_energy(current_grid):
    """Calculates total energy generated by all placed panels on the rooftop."""
    total = 0.0
    for r in range(ROWS):
        for c in range(COLS):
            if current_grid[r, c] == 1:
                adj = sum(current_grid[nr, nc] for nr, nc in get_neighbors(r, c))
                total += max(0.0, sunlight_map[r, c] - SHADOW_PENALTY * adj)
    return total

best_grid = grid.copy()
best_energy = calculate_total_energy(best_grid)
print(f"Initial Rooftop Energy Output: {best_energy:.2f} kWh/day")

# 3. MAIN PARALLEL CELLULAR ALGORITHM LOOP
coords = [(r, c) for r in range(ROWS) for c in range(COLS)]

with ThreadPoolExecutor() as executor:
    for iteration in range(MAX_ITER):
        # Step A: Evaluate all cells in parallel
        fitness_map = np.zeros((ROWS, COLS))
        results = executor.map(evaluate_cell, coords)
        for r, c, score in results:
            fitness_map[r, c] = score

        # Step B: Update cell states using neighbor information in parallel
        def update_cell(pos):
            r, c = pos
            neighbors = get_neighbors(r, c)
            best_neighbor_score = max(fitness_map[nr, nc] for nr, nc in neighbors)

            # Priority score combines the cell's own net energy and its best neighbor's influence
            priority = 0.7 * fitness_map[r, c] + 0.3 * best_neighbor_score
            return (r, c, priority)

        priority_map = np.zeros((ROWS, COLS))
        update_results = executor.map(update_cell, coords)
        for r, c, prio in update_results:
            priority_map[r, c] = prio

        # Place panels on the top-scoring cells (respecting MAX_PANELS limit)
        new_grid = np.zeros((ROWS, COLS), dtype=int)
        top_cells = np.argsort(priority_map.ravel())[::-1][:MAX_PANELS]
        new_grid.flat[top_cells] = 1

        grid = new_grid
        current_energy = calculate_total_energy(grid)

        if current_energy > best_energy:
            best_energy = current_energy
            best_grid = grid.copy()

# 4. OUTPUT THE BEST SOLUTION
print(f"Optimized Rooftop Energy Output: {best_energy:.2f} kWh/day\n")
print("Optimal Solar Panel Layout (1 = Panel, 0 = Empty):")
print(best_grid)

Initial Rooftop Energy Output: 34.87 kWh/day
Optimized Rooftop Energy Output: 38.38 kWh/day

Optimal Solar Panel Layout (1 = Panel, 0 = Empty):
[[0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1]
 [1 1 0 0 0 0 0 1]
 [1 1 1 1 0 0 0 0]
 [1 1 0 1 0 0 0 0]
 [1 0 1 1 1 1 1 1]
 [1 0 0 1 1 1 1 1]]
